# Mountain Named Entity Recognition
## Inference Demo

This notebook demonstrates how to use the trained Mountain Named Entity Recognition (NER) model.

The notebook includes:

- loading the trained model
- predicting mountain names in text
- highlighting detected entities using HTML visualization
- inference examples on both short sentences and longer texts

## Loading the trained model

The pretrained Mountain NER model is loaded using the Hugging Face pipeline API with token classification. Entity aggregation is enabled to merge consecutive tokens into complete mountain names whenever possible.

In [ ]:
from transformers import pipeline
from IPython.display import display, HTML

In [ ]:
ner = pipeline(
    "token-classification",
    model="mountain_ner_model",
    tokenizer="mountain_ner_model",
    aggregation_strategy="simple"
)

## Post-processing predictions

Although entity aggregation combines most subword tokens automatically, some mountain names may still be split into WordPiece fragments (for example, *Etna → E + ##tna*).

The following helper function merges these fragments before displaying the predictions.

In [ ]:
def merge_subwords(predictions):

    if not predictions:
        return predictions

    merged = []

    for pred in predictions:

        if pred["word"].startswith("##") and merged:

            merged[-1]["word"] += pred["word"][2:]
            merged[-1]["end"] = pred["end"]

            merged[-1]["score"] = (
                merged[-1]["score"] + pred["score"]
            ) / 2

        else:

            merged.append(pred.copy())

    return merged

## Text prediction

The `predict()` function extracts mountain names from the input text and prints each detected entity together with the model confidence score.

In [3]:
def predict(text):

    predictions = merge_subwords(ner(text))

    if not predictions:
        print("No mountains found.")
        return

    print("Detected mountains:")

    for entity in predictions:

        print(
            f"- {entity['word']} "
            f"(confidence={entity['score']:.3f})"
        )

## HTML visualization

The `highlight_mountains()` function displays the original text with detected mountain names highlighted and their confidence scores shown inline.

In [ ]:
def highlight_mountains(text):

    predictions = merge_subwords(ner(text))

    html_text = text

    for entity in sorted(predictions, key=lambda x: x["start"], reverse=True):

        start = entity["start"]
        end = entity["end"]

        word = text[start:end]

        highlighted = f"""
        <span style="
            background:#FFD54F;
            padding:2px 5px;
            border-radius:4px;
            font-weight:bold;
        ">
            {word}
        </span>
        <span style="color:#666;font-size:13px;">
            ({entity['score']:.3f})
        </span>
        """

        html_text = (
            html_text[:start]
            + highlighted
            + html_text[end:]
        )

    display(HTML(f"""
    <div style="
        font-size:18px;
        line-height:1.8;
        border:1px solid #ddd;
        border-radius:10px;
        padding:20px;
        background:white;
    ">
        {html_text}
    </div>
    """))

## Examples on short sentences

### Example 1

In [ ]:
predict("Nanda Devi, Nanga Parbat and Ojos del Salado showcase the remarkable diversity of the world's mountain landscapes, from the Himalayas to the Andes.")

Detected mountains:
- Nanda Devi (confidence=0.764)
- Nanga Parbat (confidence=0.728)
- del Salado (confidence=0.766)


In [ ]:
highlight_mountains("Nanda Devi, Nanga Parbat and Ojos del Salado showcase the remarkable diversity of the world's mountain landscapes, from the Himalayas to the Andes.")

### Example 2

In [ ]:
predict("Mount Etna is an active volcano in Italy.")

Detected mountains:
- Mount Etna (confidence=0.899)


In [ ]:
highlight_mountains("Mount Etna is an active volcano in Italy.")

## Examples on longer texts

### Example 1

In [ ]:
text1 = """
There is something magical about traveling toward a mountain you have only seen in photographs. The first glimpse of a giant peak rising above the clouds can make hours of hiking, long flights, and winding roads feel completely worthwhile. My journey through the world’s mountain landscapes began with a simple goal: to experience the places where nature feels larger than life.

My first adventure took me to the Himalayas, where the landscape seemed endless. Standing among the valleys near Mount Everest, I understood why explorers and dreamers have been drawn here for generations. The villages, prayer flags, and snowy peaks created an atmosphere that felt almost unreal. Nearby, the impressive shapes of Lhotse and Makalu reminded me how many incredible giants are hidden within this mountain range.

From Asia, my travels continued toward the dramatic landscapes of the Karakoram. My next stop was Africa, where the experience was completely different. The journey toward Mount Kilimanjaro passed through changing environments, from green forests filled with wildlife to dry landscapes near the summit. Watching the sunrise from high on the mountain was one of the most peaceful moments of my travels. In Kenya, the views around Mount Kenya offered another perspective, with unique ecosystems and breathtaking scenery near the equator.

Europe brought a new kind of mountain adventure.
"""

In [ ]:
predict(text1)

Detected mountains:
- Mount Everest (confidence=0.936)
- Mount Kiliman (confidence=0.771)
- Mount Kenya (confidence=0.918)


In [ ]:
highlight_mountains(text1)

### Example 2

In [ ]:
text2 = """
There is something magical about traveling toward a mountain you have only seen in photographs. The first glimpse of a giant peak rising above the clouds can make hours of hiking, long flights, and winding roads feel completely worthwhile. My journey through the world’s mountain landscapes began with a simple goal: to experience the places where nature feels larger than life.

My first adventure took me to the Himalayas, where the landscape seemed endless. Standing among the valleys near Mount Everest, I understood why explorers and dreamers have been drawn here for generations. The villages, prayer flags, and snowy peaks created an atmosphere that felt almost unreal. Nearby, the impressive shapes of Lhotse and Makalu reminded me how many incredible giants are hidden within this mountain range.

From Asia, my travels continued toward the dramatic landscapes of the Karakoram. Seeing K2 in the distance was unforgettable. Its sharp slopes and remote location gave it a powerful presence, while nearby Broad Peak added another layer of beauty to the surrounding scenery. This region showed me that mountains are not only places to climb but also places to admire and respect.

My next stop was Africa, where the experience was completely different. The journey toward Mount Kilimanjaro passed through changing environments, from green forests filled with wildlife to dry landscapes near the summit. Watching the sunrise from high on the mountain was one of the most peaceful moments of my travels. In Kenya, the views around Mount Kenya offered another perspective, with unique ecosystems and breathtaking scenery near the equator.

Europe brought a new kind of mountain adventure. The Alps were filled with charming villages, quiet trails, and unforgettable views. Walking beneath Mont Blanc felt like entering a postcard, while the famous Matterhorn stood proudly above the surrounding valleys. Every turn of the trail revealed another beautiful scene, from sparkling glaciers to peaceful mountain lakes.

Across the Atlantic, I explored the wild landscapes of North America. Denali impressed me with its enormous scale and remote wilderness. Later, I traveled through the forests surrounding Mount Rainier, where glaciers, rivers, and volcanic landscapes created a perfect setting for hiking. The experience reminded me that mountains are constantly changing, shaped by weather, ice, and time.

South America offered some of the most dramatic views of my journey. In Argentina, Aconcagua dominated the landscape with its massive presence. The dry valleys, colorful rocks, and high-altitude environment made the Andes feel completely different from the snowy peaks I had visited before.

Some mountains carried stories that went beyond geography. Mount Fuji in Japan was not only a beautiful volcano but also a symbol deeply connected to art, history, and culture. Watching its perfect shape appear through the morning mist was a moment I will never forget. In New Zealand, Aoraki/Mount Cook showed me a world of glaciers, alpine valleys, and powerful natural beauty.

My travels also introduced me to volcanic landscapes. The sounds and sights around Mount Etna in Italy revealed the energy hidden beneath Earth’s surface. In Indonesia, the dramatic scenery around Mount Bromo and Mount Rinjani showed how volcanoes can create some of the most fascinating environments on the planet.

Looking back on this journey, I realized that every mountain tells a different story. Some challenge climbers, some welcome hikers, and others inspire artists and travelers. Whether standing beneath the towering peaks of the Himalayas, walking through Alpine valleys, or watching volcanic landscapes change with the weather, mountains continue to remind us how incredible our planet truly is.

The adventure never really ends. Somewhere beyond the next road, trail, or horizon, another mountain is waiting to be discovered.
"""

In [ ]:
predict(text2)

Detected mountains:
- Mount Everest (confidence=0.957)
- K (confidence=0.819)
- Peak (confidence=0.696)
- Mount Kilim (confidence=0.820)
- Mount Kenya (confidence=0.964)
- Mont Blanc (confidence=0.984)


In [ ]:
highlight_mountains(text2)